# Seguimiento SAE + SIDCAR Grupo A

Este cuaderno carga las tres bases, usa el `utils.py` actualizado y genera un Excel final con 3 hojas:

- **hoja_1_grupo_a**: radicados finales del grupo A.
- **hoja_2_seguimiento**: seguimiento operativo con dummies y vencimiento.
- **hoja_3_resumen**: resumen estadístico para Power BI.


# Carga base de archivos

In [26]:
# =========================================
# SEGMENTO 1 - CARGA BASE DE LAS 3 FUENTES
# =========================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

# -----------------------------
# RUTA BASE
# -----------------------------
RUTA_ENTRADA = Path("./input_SIDCAR")

# -----------------------------
# BUSCAR ARCHIVOS POR NOMBRE
# -----------------------------
def buscar_archivo_excel(carpeta: Path, nombre_base: str):
    posibles = list(carpeta.glob(f"{nombre_base}.*"))
    posibles = [p for p in posibles if p.suffix.lower() in [".xlsx", ".xls", ".xlsm"]]
    
    if not posibles:
        raise FileNotFoundError(f"No se encontró archivo para: {nombre_base}")
    
    return posibles[0]

# -----------------------------
# CARGAR EXCEL EN BRUTO
# -----------------------------
def cargar_excel_bruto(path_archivo: Path):
    return pd.read_excel(path_archivo, header=None)

# -----------------------------
# UBICAR ARCHIVOS
# -----------------------------
archivo_act = buscar_archivo_excel(RUTA_ENTRADA, "ActividadesActuales")
archivo_pre = buscar_archivo_excel(RUTA_ENTRADA, "preradicados")
archivo_rad = buscar_archivo_excel(RUTA_ENTRADA, "Radicados")

print("Archivos encontrados:")
print("ActividadesActuales ->", archivo_act.name)
print("preradicados        ->", archivo_pre.name)
print("Radicados           ->", archivo_rad.name)

# -----------------------------
# CARGA EN BRUTO
# -----------------------------
df_act_raw = cargar_excel_bruto(archivo_act)
df_pre_raw = cargar_excel_bruto(archivo_pre)
df_rad_raw = cargar_excel_bruto(archivo_rad)

print("\nDimensiones en bruto:")
print("df_act_raw:", df_act_raw.shape)
print("df_pre_raw:", df_pre_raw.shape)
print("df_rad_raw:", df_rad_raw.shape)

Archivos encontrados:
ActividadesActuales -> ActividadesActuales.xls
preradicados        -> preradicados.xlsx
Radicados           -> Radicados.xlsx

Dimensiones en bruto:
df_act_raw: (2094, 22)
df_pre_raw: (1890, 10)
df_rad_raw: (73484, 25)


In [27]:
# =========================================================
# SEGMENTO 2 - LIMPIEZA DE TÍTULOS Y DATAFRAMES UTILIZABLES
# =========================================================

def limpiar_texto_columna(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().lower()
    texto = re.sub(r"\s+", "_", texto)
    texto = re.sub(r"[^a-z0-9_áéíóúñ]", "", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto

def hacer_columnas_unicas(columnas):
    nuevas = []
    conteo = {}
    
    for col in columnas:
        col = col if col else "columna"
        if col not in conteo:
            conteo[col] = 0
            nuevas.append(col)
        else:
            conteo[col] += 1
            nuevas.append(f"{col}_{conteo[col]}")
    
    return nuevas

def detectar_fila_header(df, revisar_hasta=10):
    mejor_fila = 0
    max_no_nulos = -1
    
    limite = min(revisar_hasta, len(df))
    
    for i in range(limite):
        cantidad = df.iloc[i].notna().sum()
        if cantidad > max_no_nulos:
            max_no_nulos = cantidad
            mejor_fila = i
    
    return mejor_fila

def limpiar_base(df_raw, nombre_base="base"):
    df_raw = df_raw.dropna(how="all").reset_index(drop=True)
    
    fila_header = detectar_fila_header(df_raw, revisar_hasta=10)
    
    headers = df_raw.iloc[fila_header].copy()
    headers = [limpiar_texto_columna(x) for x in headers]
    headers = hacer_columnas_unicas(headers)
    
    df = df_raw.iloc[fila_header + 1:].copy().reset_index(drop=True)
    df.columns = headers
    
    df = df.dropna(axis=1, how="all")
    df = df.dropna(how="all").reset_index(drop=True)
    
    print(f"\nBase limpia: {nombre_base}")
    print("Fila header detectada:", fila_header)
    print("Dimensión final:", df.shape)
    print("Columnas:")
    print(list(df.columns))
    
    return df

# -----------------------------
# LIMPIEZA DE LAS 3 BASES
# -----------------------------
df_act = limpiar_base(df_act_raw, "ActividadesActuales")
df_pre = limpiar_base(df_pre_raw, "preradicados")
df_rad = limpiar_base(df_rad_raw, "Radicados")

# -----------------------------
# VISTA RÁPIDA
# -----------------------------
print("\nVista rápida ActividadesActuales")
display(df_act.head(3))

print("\nVista rápida preradicados")
display(df_pre.head(3))

print("\nVista rápida Radicados")
display(df_rad.head(3))


Base limpia: ActividadesActuales
Fila header detectada: 1
Dimensión final: (2092, 22)
Columnas:
['número', 'tipo_de_trámite', 'trámite', 'estado_actual', 'regional', 'pac', 'fecha_apertura', 'nombre', 'fecha_asignación', 'estado_usuario', 'días_responsable', 'formatoexp', 'etapa', 'actividad', 'prerad', 'estado', 'devoluciones', 'radicado', 'nombre_1', 'fecha_asignación_1', 'días_responsable_1', 'comentarios']

Base limpia: preradicados
Fila header detectada: 2
Dimensión final: (1887, 10)
Columnas:
['3669770', 'oficio', 'en_borrador', '2026', '20260105_142543290000', 'nrosas', 'nancy_yaneth_rosas', 'dirección_regional_ubaté', 'notificación_por_aviso_resolución_dja_no_00961_de_2025_expediente_58013', 'columna']

Base limpia: Radicados
Fila header detectada: 0
Dimensión final: (73482, 23)
Columnas:
['no_doc', 'fecha_correo', 'radicado', 'tipo', 'f_radicado', 'asunto', 'título', 'elaboró', 'revisó', 'radicó', 'of_radica', 'estado', 'medio', 'tramite', 'destinatarios', 'remitente', 'respo

,número,tipo_de_trámite,trámite,estado_actual,regional,pac,fecha_apertura,nombre,fecha_asignación,estado_usuario,...,etapa,actividad,prerad,estado,devoluciones,radicado,nombre_1,fecha_asignación_1,días_responsable_1,comentarios
0,51959,Permisivo,Autorización de Ocupación de Cauces y/o obras ...,Seguimiento y Control,Magdalena Centro,Actividad 12.2.5,2015-10-27 09:43:36.197000,Milena Alvarez Dussan (DJA),2026-04-05 16:58:05.780000,Activo,...,Seguimiento y Control,Acto Administrativo,713456,Enviado para Revisión,NaN,NaN,Alvaro Andres Jimenez Vanegas (DJA),2026-03-17 10:20:56,10,NaN
1,16292,Permisivo,Licencia Ambiental,Seguimiento y Control,Sabana Occidente,Actividad 12.2.5,2000-08-15 00:00:00,Milena Alvarez Dussan (DJA),2026-04-05 15:52:03.943000,Activo,...,Seguimiento y Control,Acto Administrativo,713446,Enviado para Revisión,NaN,NaN,Alvaro Andres Jimenez Vanegas (DJA),2026-03-02 23:16:30,21,NaN
2,69507,Sancionatorio,Afectación recurso Agua,Seguimiento y Control,Chiquinquira,Actividad 12.2.5,2018-06-28 16:45:22.027000,Emma Constanza Zuñiga Galvis (DJA),2026-04-05 15:32:48.723000,Activo,...,Seguimiento a la Sanción Impuesta,Reparto Técnico,685078,Enviado para Aprobación,1,NaN,Joaquin Fernando Sarmiento Cardenas (DJA),2026-02-24 14:51:42.603000,25,Revisado y ajustado y con decisión de Rechazar...



Vista rápida preradicados


,3669770,oficio,en_borrador,2026,20260105_142543290000,nrosas,nancy_yaneth_rosas,dirección_regional_ubaté,notificación_por_aviso_resolución_dja_no_00961_de_2025_expediente_58013,columna
0,3670114,Memorando,En Borrador,2026,2026-01-05 17:11:35.850000,nrosas,Nancy Yaneth Rosas,Dirección Regional Ubaté,Solicitud dar prioridad a trámites asignados.,NaN
1,3670867,Oficio,En Borrador,2026,2026-01-06 14:26:52.433000,marevalo,Miryam Cecilia Arevalo,Dirección Regional Almeidas y Guatavita,Notificación por Aviso Auto DRAG No. 00474 de ...,NaN
2,3670942,Oficio,En Borrador,2026,2026-01-06 15:16:01.477000,marevalo,Miryam Cecilia Arevalo,Dirección Regional Almeidas y Guatavita,Notificación por Aviso Resolución DRAG No. 000...,NaN



Vista rápida Radicados


,no_doc,fecha_correo,radicado,tipo,f_radicado,asunto,título,elaboró,revisó,radicó,...,tramite,destinatarios,remitente,responsable,preradicado,tipo_asunto,registropor,info_sae,columna_1,estado_términos
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Expediente,Proceso,NaN
1,NaN,2026-01-02 05:55:27.367000,12262000001,Oficio,2026-01-02 05:55:27.250000,Solicitud Información Capacidad Socioeconómica...,N.A,Cristian Camilo Romero Gualdron,NaN,Erika Alvarez Castañeda,...,NaN,DEPARTAMENTO NACIONAL DE PLANEACION DNP,Dirección Regional Sumapaz (Erika Alvarez Cast...,NaN,NaN,NaN,ealvarezca,NaN,NaN,NaN
2,NaN,2026-01-02 05:56:27.737000,12262000002,Oficio,2026-01-02 05:56:27.730000,Respuesta radicado 20251121707: Arbol en Riesg...,N.A,Alfonso Pulido Moreno,NaN,Erika Alvarez Castañeda,...,NaN,FABIO LOZANO TORRES(ALCALDIA DE FUSAGASUGA),Dirección Regional Sumapaz (Erika Alvarez Cast...,NaN,NaN,NaN,ealvarezca,NaN,NaN,NaN


# Tratado de base radicados
## Aca sacamos el grupo I del tablero a proyectar 

In [28]:
# =========================================================
# SEGMENTO 3 - ARMAR CUADRO FINAL ORDENADO
# =========================================================

import pandas as pd
import numpy as np
import re
import unicodedata

# ---------------------------------------------------------
# 1) ABOGADOS DEL GRUPO
# ---------------------------------------------------------
NOMBRES_ABOGADOS = [
    "FRANCY LILIANA DAZA SANCHEZ",
    "Maria Fernanda Arias Oyuela",
    "Isabel Cristina Angarita Perpiñán",
    "Gina Alexandra Roberto Torres",
    "Alvaro Andres Jimenez Vanegas",
]

# ---------------------------------------------------------
# 2) FUNCIONES AUXILIARES
# ---------------------------------------------------------
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def normalizar_abogado(texto):
    texto = normalizar_texto(texto)
    texto = re.split(r"/|\(", texto)[0].strip()
    return texto

def extraer_expediente(texto):
    if pd.isna(texto):
        return np.nan

    texto = normalizar_texto(texto)

    patrones = [
        r"NUMERO EXPEDIENTE\s*:\s*(\d+)",
        r"NÚMERO EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s+(\d+)",
    ]

    for patron in patrones:
        m = re.search(patron, texto)
        if m:
            return m.group(1)

    return np.nan

def clasificar_asunto(asunto, titulo):
    texto = normalizar_texto(f"{asunto} {titulo}")
    if "NOTIFIC" in texto:
        return "GESTION"
    return "IMPULSO"

def extraer_regional(texto):
    if pd.isna(texto):
        return ""
    texto = normalizar_texto(texto)
    texto = re.split(r"\(|-|/", texto)[0].strip()
    return texto

# ---------------------------------------------------------
# 3) NORMALIZAR GRUPO
# ---------------------------------------------------------
abogados_norm = [normalizar_abogado(x) for x in NOMBRES_ABOGADOS]

# ---------------------------------------------------------
# 4) PREPARAR RADICADOS DESDE df_rad
# ---------------------------------------------------------
df_rad_base = df_rad.copy()

df_rad_base["abogado_norm"] = df_rad_base["elaboró"].apply(normalizar_abogado)

df_rad_base["expediente"] = df_rad_base["asunto"].apply(extraer_expediente)

mask_sin_exp = df_rad_base["expediente"].isna()
df_rad_base.loc[mask_sin_exp, "expediente"] = df_rad_base.loc[mask_sin_exp, "título"].apply(extraer_expediente)

df_rad_base["regional_norm"] = df_rad_base["remitente"].apply(extraer_regional)

df_rad_base["estado_asunto"] = df_rad_base.apply(
    lambda row: clasificar_asunto(row.get("asunto", ""), row.get("título", "")),
    axis=1
)

# ---------------------------------------------------------
# 5) FILTRAR SOLO EL GRUPO DE ABOGADOS
# ---------------------------------------------------------
df_cuadro_radicados = df_rad_base[
    (df_rad_base["abogado_norm"].isin(abogados_norm)) &
    (df_rad_base["expediente"].notna())
].copy()

# ---------------------------------------------------------
# 6) SELECCIONAR SOLO LO NECESARIO
# ---------------------------------------------------------
df_cuadro_radicados = df_cuadro_radicados[
    [
        "abogado_norm",
        "expediente",
        "regional_norm",
        "estado_asunto",
        "asunto",
        "título",
        "f_radicado"
    ]
].copy()

# ---------------------------------------------------------
# 7) RENOMBRAR COLUMNAS FINALES
# ---------------------------------------------------------
df_cuadro_radicados = df_cuadro_radicados.rename(columns={
    "abogado_norm": "abogado_normalizado",
    "regional_norm": "regional",
    "estado_asunto": "clasificacion_asunto",
    "asunto": "asunto_original",
    "título": "titulo_original",
    "f_radicado": "fecha_radicado"
})

# ---------------------------------------------------------
# 8) LIMPIAR FORMATO
# ---------------------------------------------------------
df_cuadro_radicados["abogado_normalizado"] = df_cuadro_radicados["abogado_normalizado"].astype(str).str.strip()
df_cuadro_radicados["expediente"] = df_cuadro_radicados["expediente"].astype(str).str.strip()
df_cuadro_radicados["regional"] = df_cuadro_radicados["regional"].astype(str).str.strip()
df_cuadro_radicados["clasificacion_asunto"] = df_cuadro_radicados["clasificacion_asunto"].astype(str).str.strip()
df_cuadro_radicados["fecha_radicado"] = pd.to_datetime(df_cuadro_radicados["fecha_radicado"], errors="coerce")

# ---------------------------------------------------------
# 9) ORDENAR
# ---------------------------------------------------------
df_cuadro_radicados = df_cuadro_radicados.sort_values(
    by=["abogado_normalizado", "expediente", "fecha_radicado"],
    ascending=[True, True, True]
).reset_index(drop=True)

print("Total registros finales con expediente:", len(df_cuadro_radicados))
display(df_cuadro_radicados)

Total registros finales con expediente: 102


,abogado_normalizado,expediente,regional,clasificacion_asunto,asunto_original,titulo_original,fecha_radicado
0,ALVARO ANDRES JIMENEZ VANEGAS,102134,DIRECCION REGIONAL TEQUENDAMA,IMPULSO,Por el cual se hace requerimiento y cobro por ...,Por el cual se hace requerimiento y cobro por ...,2026-03-05 16:20:01.730
1,ALVARO ANDRES JIMENEZ VANEGAS,102134,DIRECCION REGIONAL TEQUENDAMA,GESTION,Notificación personal medio electrónico Auto D...,N.A,2026-03-13 11:20:27.977
2,ALVARO ANDRES JIMENEZ VANEGAS,102134,DIRECCION REGIONAL TEQUENDAMA,GESTION,Notificación personal medio electrónico Auto D...,N.A,2026-03-13 11:20:27.977
3,ALVARO ANDRES JIMENEZ VANEGAS,33512,DIRECCION REGIONAL SABANA OCCIDENTE,IMPULSO,Por medio de la cual se declara la pérdida de ...,Por medio de la cual se declara la pérdida de ...,2026-03-27 09:56:02.127
4,ALVARO ANDRES JIMENEZ VANEGAS,57966,DIRECCION REGIONAL TEQUENDAMA,IMPULSO,Por el cual se anuncia la caducidad de una con...,Por el cual se anuncia la caducidad de una con...,2026-03-10 13:20:18.303
...,...,...,...,...,...,...,...
97,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,Expediente 98496. Citación notificación personal.,N.A,2026-03-13 09:58:51.790
98,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,Expediente 98496. Citación notificación personal.,N.A,2026-03-13 09:58:51.790
99,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,Expediente 98496. Citación notificación personal.,N.A,2026-03-13 09:59:37.207
100,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,Expediente 98496. Citación notificación personal.,N.A,2026-03-13 09:59:37.207


In [29]:
# =========================================================
# SEGMENTO 4 - VER RESULTADO ORDENADO EN PANTALLA
# =========================================================

print("\nCUADRO FINAL ORDENADO:")
display(
    df_cuadro_radicados[
        [
            "abogado_normalizado",
            "expediente",
            "regional",
            "clasificacion_asunto",
            "fecha_radicado",
            "asunto_original",
            "titulo_original"
        ]
    ]
)

print("\nRESUMEN POR CLASIFICACIÓN:")
display(
    df_cuadro_radicados.groupby("clasificacion_asunto", dropna=False)
    .size()
    .reset_index(name="cantidad")
)

print("\nRESUMEN POR ABOGADO:")
display(
    df_cuadro_radicados.groupby("abogado_normalizado", dropna=False)
    .size()
    .reset_index(name="cantidad")
)


CUADRO FINAL ORDENADO:


,abogado_normalizado,expediente,regional,clasificacion_asunto,fecha_radicado,asunto_original,titulo_original
0,ALVARO ANDRES JIMENEZ VANEGAS,102134,DIRECCION REGIONAL TEQUENDAMA,IMPULSO,2026-03-05 16:20:01.730,Por el cual se hace requerimiento y cobro por ...,Por el cual se hace requerimiento y cobro por ...
1,ALVARO ANDRES JIMENEZ VANEGAS,102134,DIRECCION REGIONAL TEQUENDAMA,GESTION,2026-03-13 11:20:27.977,Notificación personal medio electrónico Auto D...,N.A
2,ALVARO ANDRES JIMENEZ VANEGAS,102134,DIRECCION REGIONAL TEQUENDAMA,GESTION,2026-03-13 11:20:27.977,Notificación personal medio electrónico Auto D...,N.A
3,ALVARO ANDRES JIMENEZ VANEGAS,33512,DIRECCION REGIONAL SABANA OCCIDENTE,IMPULSO,2026-03-27 09:56:02.127,Por medio de la cual se declara la pérdida de ...,Por medio de la cual se declara la pérdida de ...
4,ALVARO ANDRES JIMENEZ VANEGAS,57966,DIRECCION REGIONAL TEQUENDAMA,IMPULSO,2026-03-10 13:20:18.303,Por el cual se anuncia la caducidad de una con...,Por el cual se anuncia la caducidad de una con...
...,...,...,...,...,...,...,...
97,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,2026-03-13 09:58:51.790,Expediente 98496. Citación notificación personal.,N.A
98,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,2026-03-13 09:58:51.790,Expediente 98496. Citación notificación personal.,N.A
99,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,2026-03-13 09:59:37.207,Expediente 98496. Citación notificación personal.,N.A
100,MARIA FERNANDA ARIAS OYUELA,98496,DIRECCION REGIONAL ALMEIDAS Y GUATAVITA,GESTION,2026-03-13 09:59:37.207,Expediente 98496. Citación notificación personal.,N.A



RESUMEN POR CLASIFICACIÓN:


,clasificacion_asunto,cantidad
0,GESTION,35
1,IMPULSO,67



RESUMEN POR ABOGADO:


,abogado_normalizado,cantidad
0,ALVARO ANDRES JIMENEZ VANEGAS,18
1,FRANCY LILIANA DAZA SANCHEZ,1
2,GINA ALEXANDRA ROBERTO TORRES,27
3,ISABEL CRISTINA ANGARITA PERPINAN,29
4,MARIA FERNANDA ARIAS OYUELA,27


# Tratasmiento base Etapas y actividades
## Nos centramos ahora en analizar la base SAE para conocer que etapas y actividades hay 

In [30]:
# =========================================================
# SEGMENTO 5 - ACTIVIDADES ACTUALES + CRUCE PRE/RAD
#            + CLASIFICACION DE ASUNTO SIN VACIOS
# =========================================================

import pandas as pd
import numpy as np
import re
import unicodedata

# ---------------------------------------------------------
# 1) FUNCIONES BASE
# ---------------------------------------------------------
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def normalizar_nombre_persona(texto):
    texto = normalizar_texto(texto)
    texto = re.split(r"/|\(", texto)[0].strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def limpiar_expediente(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip()
    m = re.search(r"(\d+)", texto)
    return m.group(1) if m else np.nan

def extraer_expediente_texto(texto):
    if pd.isna(texto):
        return np.nan
    
    texto = str(texto)
    texto_may = normalizar_texto(texto)

    patrones = [
        r"NUMERO EXPEDIENTE\s*:\s*(\d+)",
        r"NÚMERO EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s*(\d+)",
    ]
    
    for patron in patrones:
        m = re.search(patron, texto_may)
        if m:
            return m.group(1)
    
    return np.nan

def extraer_regional_simple(texto):
    if pd.isna(texto):
        return np.nan
    texto = normalizar_texto(texto)
    texto = re.split(r" - |\(|/", texto)[0].strip()
    return texto

def clasificar_asunto_sidcar(asunto, titulo):
    """
    GESTION = notificaciones
    IMPULSO = todo lo demás
    """
    texto = f"{asunto} {titulo}"
    texto = normalizar_texto(texto)

    if "NOTIFIC" in texto:
        return "GESTION"
    return "IMPULSO"

# ---------------------------------------------------------
# 2) LISTA DE ABOGADOS OBJETIVO
# ---------------------------------------------------------
NOMBRES_ABOGADOS = [
    "FRANCY LILIANA DAZA SANCHEZ",
    "Maria Fernanda Arias Oyuela",
    "Isabel Cristina Angarita Perpiñán",
    "Gina Alexandra Roberto Torres",
    "Alvaro Andres Jimenez Vanegas",
]

ABOGADOS_OBJETIVO_NORM = [normalizar_nombre_persona(x) for x in NOMBRES_ABOGADOS]

# ---------------------------------------------------------
# 3) MAPEO DE COLUMNAS EN ACTIVIDADES ACTUALES
# ---------------------------------------------------------
col_expediente_act   = df_act.columns[0]   # A
col_regional_act     = df_act.columns[4]   # E
col_responsable_act  = df_act.columns[7]   # H
col_estado_act       = df_act.columns[15]  # P
col_devoluciones_act = df_act.columns[16]  # Q
col_remite_act       = df_act.columns[18]  # S

columnas_fecha = [c for c in df_act.columns if "fecha" in str(c).lower()]
col_fecha_act = columnas_fecha[0] if columnas_fecha else None

print("Columnas tomadas en ActividadesActuales:")
print("Expediente  ->", col_expediente_act)
print("Regional    ->", col_regional_act)
print("Responsable ->", col_responsable_act)
print("Estado      ->", col_estado_act)
print("Devolucion  ->", col_devoluciones_act)
print("Remite      ->", col_remite_act)
print("Fecha       ->", col_fecha_act)

# ---------------------------------------------------------
# 4) PREPARAR ACTIVIDADES ACTUALES
# ---------------------------------------------------------
df_act_base = df_act.copy()

df_act_base["expediente"] = df_act_base[col_expediente_act].apply(limpiar_expediente)
df_act_base["regional"] = df_act_base[col_regional_act].apply(extraer_regional_simple)
df_act_base["responsable_norm"] = df_act_base[col_responsable_act].apply(normalizar_nombre_persona)
df_act_base["remite_norm"] = df_act_base[col_remite_act].apply(normalizar_nombre_persona)
df_act_base["estado_sidcar"] = df_act_base[col_estado_act].astype(str).fillna("").str.strip()
df_act_base["devolucion_sidcar"] = df_act_base[col_devoluciones_act].astype(str).fillna("").str.strip()

if col_fecha_act:
    df_act_base["fecha_asignacion"] = pd.to_datetime(df_act_base[col_fecha_act], errors="coerce")
else:
    df_act_base["fecha_asignacion"] = pd.NaT

df_act_base = df_act_base[df_act_base["expediente"].notna()].copy()

df_act_base = df_act_base[
    (df_act_base["responsable_norm"].isin(ABOGADOS_OBJETIVO_NORM)) |
    (df_act_base["remite_norm"].isin(ABOGADOS_OBJETIVO_NORM))
].copy()

print("\nRegistros filtrados en ActividadesActuales:", len(df_act_base))

# ---------------------------------------------------------
# 5) PREPARAR PRERADICADOS
# ---------------------------------------------------------
df_pre_base = df_pre.copy()
df_pre_base["expediente"] = np.nan

if "asunto" in df_pre_base.columns:
    df_pre_base["expediente"] = df_pre_base["asunto"].apply(extraer_expediente_texto)

if "título" in df_pre_base.columns:
    mask_pre = df_pre_base["expediente"].isna()
    df_pre_base.loc[mask_pre, "expediente"] = df_pre_base.loc[mask_pre, "título"].apply(extraer_expediente_texto)

df_pre_base = df_pre_base[df_pre_base["expediente"].notna()].copy()
df_pre_base["clasificacion_asunto"] = df_pre_base.apply(
    lambda row: clasificar_asunto_sidcar(row.get("asunto", ""), row.get("título", "")),
    axis=1
)
df_pre_base["tiene_preradicado"] = 1

df_pre_resumen = (
    df_pre_base.groupby("expediente", as_index=False)
    .agg(tiene_preradicado=("tiene_preradicado", "max"))
)

df_pre_asuntos = df_pre_base[["expediente", "clasificacion_asunto"]].drop_duplicates().copy()

# ---------------------------------------------------------
# 6) PREPARAR RADICADOS
# ---------------------------------------------------------
df_rad_base = df_rad.copy()
df_rad_base["expediente"] = np.nan

if "asunto" in df_rad_base.columns:
    df_rad_base["expediente"] = df_rad_base["asunto"].apply(extraer_expediente_texto)

if "título" in df_rad_base.columns:
    mask_rad = df_rad_base["expediente"].isna()
    df_rad_base.loc[mask_rad, "expediente"] = df_rad_base.loc[mask_rad, "título"].apply(extraer_expediente_texto)

df_rad_base = df_rad_base[df_rad_base["expediente"].notna()].copy()
df_rad_base["clasificacion_asunto"] = df_rad_base.apply(
    lambda row: clasificar_asunto_sidcar(row.get("asunto", ""), row.get("título", "")),
    axis=1
)
df_rad_base["tiene_radicado"] = 1

df_rad_resumen = (
    df_rad_base.groupby("expediente", as_index=False)
    .agg(tiene_radicado=("tiene_radicado", "max"))
)

df_rad_asuntos = df_rad_base[["expediente", "clasificacion_asunto"]].drop_duplicates().copy()

# ---------------------------------------------------------
# 7) UNIFICAR ASUNTOS SIDCAR
# ---------------------------------------------------------
df_asuntos_sidcar = pd.concat(
    [df_pre_asuntos, df_rad_asuntos],
    ignore_index=True
).drop_duplicates().copy()

# ---------------------------------------------------------
# 8) MERGE GENERAL
# ---------------------------------------------------------
df_act_merge = df_act_base.merge(df_pre_resumen, on="expediente", how="left")
df_act_merge = df_act_merge.merge(df_rad_resumen, on="expediente", how="left")
df_act_merge = df_act_merge.merge(df_asuntos_sidcar, on="expediente", how="left")

df_act_merge["tiene_preradicado"] = df_act_merge["tiene_preradicado"].fillna(0).astype(int)
df_act_merge["tiene_radicado"] = df_act_merge["tiene_radicado"].fillna(0).astype(int)

# no dejamos nada sin clasificar
df_act_merge["clasificacion_asunto"] = df_act_merge["clasificacion_asunto"].fillna("IMPULSO")

# columna visual aparte de preradicado
df_act_merge["preradicado_estado"] = np.where(
    df_act_merge["tiene_preradicado"] == 1,
    "SI",
    "NO"
)

print("\nDimensión después del merge:", df_act_merge.shape)
display(df_act_merge.head(10))

Columnas tomadas en ActividadesActuales:
Expediente  -> número
Regional    -> regional
Responsable -> nombre
Estado      -> estado
Devolucion  -> devoluciones
Remite      -> nombre_1
Fecha       -> fecha_apertura

Registros filtrados en ActividadesActuales: 164

Dimensión después del merge: (177, 32)


,número,tipo_de_trámite,trámite,estado_actual,regional,pac,fecha_apertura,nombre,fecha_asignación,estado_usuario,...,expediente,responsable_norm,remite_norm,estado_sidcar,devolucion_sidcar,fecha_asignacion,tiene_preradicado,tiene_radicado,clasificacion_asunto,preradicado_estado
0,51959,Permisivo,Autorización de Ocupación de Cauces y/o obras ...,Seguimiento y Control,MAGDALENA CENTRO,Actividad 12.2.5,2015-10-27 09:43:36.197000,Milena Alvarez Dussan (DJA),2026-04-05 16:58:05.780000,Activo,...,51959,MILENA ALVAREZ DUSSAN,ALVARO ANDRES JIMENEZ VANEGAS,Enviado para Revisión,nan,2015-10-27 09:43:36.197,0,0,IMPULSO,NO
1,16292,Permisivo,Licencia Ambiental,Seguimiento y Control,SABANA OCCIDENTE,Actividad 12.2.5,2000-08-15 00:00:00,Milena Alvarez Dussan (DJA),2026-04-05 15:52:03.943000,Activo,...,16292,MILENA ALVAREZ DUSSAN,ALVARO ANDRES JIMENEZ VANEGAS,Enviado para Revisión,nan,2000-08-15 00:00:00.000,0,0,IMPULSO,NO
2,96479,Permisivo,Concesión de Aguas Superficiales,Seguimiento y Control,TEQUENDAMA,Actividad 12.2.5,2022-11-28 00:19:42.707000,Alvaro Andres Jimenez Vanegas (DJA),2026-04-05 08:21:30.650000,Activo,...,96479,ALVARO ANDRES JIMENEZ VANEGAS,MARIA FERNANDA ARIAS OYUELA,Enviado para Revisión,1,2022-11-28 00:19:42.707,0,0,IMPULSO,NO
3,54872,Permisivo,Concesión de Aguas Superficiales,Seguimiento y Control,TEQUENDAMA,Actividad 12.2.5,2016-04-25 17:52:32.540000,Alvaro Andres Jimenez Vanegas (DJA),2026-04-04 16:09:02.110000,Activo,...,54872,ALVARO ANDRES JIMENEZ VANEGAS,MARIA FERNANDA ARIAS OYUELA,Enviado para Revisión,1,2016-04-25 17:52:32.540,0,0,IMPULSO,NO
4,54194,Permisivo,Aprovechamiento Forestal Persistente Bosque Na...,Seguimiento y Control,RIONEGRO,Actividad 12.2.5,2016-03-16 16:20:02.753000,ANGI TATIANA CUERVO DAZA (DJA),2026-04-04 13:49:32.190000,Activo,...,54194,ANGI TATIANA CUERVO DAZA,FRANCY LILIANA DAZA SANCHEZ,Enviado para Revisión,nan,2016-03-16 16:20:02.753,0,0,IMPULSO,NO
5,82454,Permisivo,Aprovechamiento Forestal Bosque Natural Doméstico,Seguimiento y Control,SABANA OCCIDENTE,Actividad 12.2.5,2020-07-28 22:51:34.707000,Milena Alvarez Dussan (DJA),2026-04-03 18:28:29.053000,Activo,...,82454,MILENA ALVAREZ DUSSAN,ALVARO ANDRES JIMENEZ VANEGAS,Enviado para Revisión,nan,2020-07-28 22:51:34.707,0,0,IMPULSO,NO
6,54989,Permisivo,Concesión de Aguas Superficiales,Seguimiento y Control,TEQUENDAMA,Actividad 12.2.5,2016-04-29 16:12:56.907000,Alvaro Andres Jimenez Vanegas (DJA),2026-04-02 18:32:26.850000,Activo,...,54989,ALVARO ANDRES JIMENEZ VANEGAS,MARIA FERNANDA ARIAS OYUELA,Enviado para Revisión,1,2016-04-29 16:12:56.907,0,0,IMPULSO,NO
7,64173,Permisivo,Aprovechamiento forestal árboles aislados,Seguimiento y Control,MAGDALENA CENTRO,Actividad 12.2.5,2017-09-05 15:45:41.277000,Milena Alvarez Dussan (DJA),2026-03-31 14:25:33.063000,Activo,...,64173,MILENA ALVAREZ DUSSAN,ALVARO ANDRES JIMENEZ VANEGAS,Enviado para Revisión,nan,2017-09-05 15:45:41.277,0,1,IMPULSO,NO
8,76557,Permisivo,Aprovechamiento Forestal Único Bosque Natural,Seguimiento y Control,SABANA OCCIDENTE,Actividad 12.2.5,2019-07-11 15:58:43.227000,Milena Alvarez Dussan (DJA),2026-03-31 13:28:16.903000,Activo,...,76557,MILENA ALVAREZ DUSSAN,ALVARO ANDRES JIMENEZ VANEGAS,Enviado para Revisión,nan,2019-07-11 15:58:43.227,0,0,IMPULSO,NO
9,8552,Permisivo,Licencia Ambiental,Seguimiento y Control,SUMAPAZ,Actividad 12.2.5,1997-09-09 00:00:00,Hernan Mauricio Medina Forero (DJA),2026-03-31 12:22:50.993000,Activo,...,8552,HERNAN MAURICIO MEDINA FORERO,ISABEL CRISTINA ANGARITA PERPINAN,Enviado para Revisión,nan,1997-09-09 00:00:00.000,0,0,IMPULSO,NO


In [31]:
# =========================================================
# SEGMENTO 6 - BASE FINAL SIN FINALIZADOS
#            + PRERADICADO APARTE
#            + CLASIFICACION APARTE
# =========================================================

# ---------------------------------------------------------
# 1) FUNCIONES DE ESTADO
# ---------------------------------------------------------
def tiene_texto_valido(x):
    if pd.isna(x):
        return False
    x = str(x).strip().upper()
    return x not in ["", "NAN", "NONE", "0"]

def clasificar_pelota_caliente(row):
    estado_txt = normalizar_texto(row.get("estado_sidcar", ""))
    devol_txt = normalizar_texto(row.get("devolucion_sidcar", ""))
    tiene_pre = int(row.get("tiene_preradicado", 0))
    remite = row.get("remite_norm", "")
    
    if tiene_pre == 1 and (
        "DEVUELT" in estado_txt or
        "DEVUELT" in devol_txt or
        tiene_texto_valido(devol_txt)
    ):
        return "DEVUELTO"
    
    if tiene_pre == 1 and remite in ABOGADOS_OBJETIVO_NORM:
        return "APROBADO VOBO"
    
    if tiene_pre == 1:
        return "EN CREACION"
    
    return "SIN INICIAR"

def definir_abogado_pelota(row):
    estado = row.get("estado_pelota_caliente", "")
    responsable = row.get("responsable_norm", "")
    remite = row.get("remite_norm", "")
    
    if estado == "APROBADO VOBO" and remite in ABOGADOS_OBJETIVO_NORM:
        return remite
    
    if responsable in ABOGADOS_OBJETIVO_NORM:
        return responsable
    
    if remite in ABOGADOS_OBJETIVO_NORM:
        return remite
    
    return responsable if responsable else remite

def calcular_dias(row):
    fecha = row.get("fecha_asignacion", pd.NaT)
    if pd.isna(fecha):
        return np.nan
    hoy = pd.Timestamp.today().normalize()
    return (hoy - pd.to_datetime(fecha).normalize()).days

def estado_del_dia(dias):
    if pd.isna(dias):
        return "SIN FECHA"
    if dias <= 25:
        return "EN TRAMITE"
    elif 26 <= dias <= 30:
        return "POR VENCER"
    else:
        return "VENCIDO"

def prioridad_estado(estado):
    prioridades = {
        "APROBADO VOBO": 4,
        "DEVUELTO": 3,
        "EN CREACION": 2,
        "SIN INICIAR": 1,
    }
    return prioridades.get(estado, 0)

# ---------------------------------------------------------
# 2) EXCLUIR RADICADOS / FINALIZADOS
# ---------------------------------------------------------
df_estado = df_act_merge.copy()

df_estado = df_estado[df_estado["tiene_radicado"] == 0].copy()

# ---------------------------------------------------------
# 3) CONSTRUIR CAMPOS
# ---------------------------------------------------------
df_estado["estado_pelota_caliente"] = df_estado.apply(clasificar_pelota_caliente, axis=1)
df_estado["abogado_normalizado"] = df_estado.apply(definir_abogado_pelota, axis=1)
df_estado["dias_transcurridos"] = df_estado.apply(calcular_dias, axis=1)
df_estado["estado_del_dia"] = df_estado["dias_transcurridos"].apply(estado_del_dia)
df_estado["prioridad_estado"] = df_estado["estado_pelota_caliente"].apply(prioridad_estado)

# por seguridad, nunca dejar sin clasificar
df_estado["clasificacion_asunto"] = df_estado["clasificacion_asunto"].fillna("IMPULSO")
df_estado["clasificacion_asunto"] = df_estado["clasificacion_asunto"].replace("", "IMPULSO")

# ---------------------------------------------------------
# 4) UNA FILA POR EXPEDIENTE + CLASIFICACION
# ---------------------------------------------------------
df_estado = df_estado.sort_values(
    by=["expediente", "clasificacion_asunto", "prioridad_estado", "dias_transcurridos"],
    ascending=[True, True, False, False]
).copy()

df_base_final_pelota = df_estado.drop_duplicates(
    subset=["expediente", "clasificacion_asunto"],
    keep="first"
).copy()

# ---------------------------------------------------------
# 5) COLUMNAS FINALES
# ---------------------------------------------------------
df_base_final_pelota = df_base_final_pelota[
    [
        "abogado_normalizado",
        "expediente",
        "regional",
        "preradicado_estado",
        "clasificacion_asunto",
        "estado_pelota_caliente",
        "dias_transcurridos",
        "estado_del_dia",
        "responsable_norm",
        "remite_norm",
        "estado_sidcar",
        "devolucion_sidcar",
        "fecha_asignacion"
    ]
].copy()

df_base_final_pelota = df_base_final_pelota.sort_values(
    by=["abogado_normalizado", "expediente", "clasificacion_asunto"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# 6) VISUALIZACION
# ---------------------------------------------------------
print("Total filas finales sin radicados/finalizados:", len(df_base_final_pelota))

print("\nBASE FINAL DEPURADA:")
display(df_base_final_pelota)

print("\nRESUMEN POR PRERADICADO:")
display(
    df_base_final_pelota.groupby("preradicado_estado", dropna=False)
    .size()
    .reset_index(name="cantidad")
)

print("\nRESUMEN POR CLASIFICACION DE ASUNTO:")
display(
    df_base_final_pelota.groupby("clasificacion_asunto", dropna=False)
    .size()
    .reset_index(name="cantidad")
)

print("\nRESUMEN POR ESTADO:")
display(
    df_base_final_pelota.groupby("estado_pelota_caliente", dropna=False)
    .size()
    .reset_index(name="cantidad")
)

Total filas finales sin radicados/finalizados: 119

BASE FINAL DEPURADA:


,abogado_normalizado,expediente,regional,preradicado_estado,clasificacion_asunto,estado_pelota_caliente,dias_transcurridos,estado_del_dia,responsable_norm,remite_norm,estado_sidcar,devolucion_sidcar,fecha_asignacion
0,ALVARO ANDRES JIMENEZ VANEGAS,100651,ALMEIDAS Y MUNICIPIO DE GUATAVITA,NO,IMPULSO,SIN INICIAR,1013,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-06-27 08:11:04.867
1,ALVARO ANDRES JIMENEZ VANEGAS,101017,SABANA OCCIDENTE,NO,IMPULSO,SIN INICIAR,997,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-07-13 11:16:46.047
2,ALVARO ANDRES JIMENEZ VANEGAS,101302,ALMEIDAS Y MUNICIPIO DE GUATAVITA,NO,IMPULSO,SIN INICIAR,984,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-07-26 16:24:54.210
3,ALVARO ANDRES JIMENEZ VANEGAS,102916,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,916,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-10-02 17:12:17.327
4,ALVARO ANDRES JIMENEZ VANEGAS,102917,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,916,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-10-02 17:12:38.233
...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,MARIA FERNANDA ARIAS OYUELA,57888,BAJO MAGDALENA,NO,IMPULSO,SIN INICIAR,3489,VENCIDO,MARIA FERNANDA ARIAS OYUELA,WILSON LEONARDO JIMENEZ CASTILLO,nan,nan,2016-09-15 12:14:58.250
115,MARIA FERNANDA ARIAS OYUELA,57959,BAJO MAGDALENA,NO,IMPULSO,SIN INICIAR,3484,VENCIDO,MARIA FERNANDA ARIAS OYUELA,WILSON LEONARDO JIMENEZ CASTILLO,nan,nan,2016-09-20 09:39:07.440
116,MARIA FERNANDA ARIAS OYUELA,57960,BAJO MAGDALENA,NO,IMPULSO,SIN INICIAR,3484,VENCIDO,MARIA FERNANDA ARIAS OYUELA,WILSON LEONARDO JIMENEZ CASTILLO,nan,nan,2016-09-20 09:44:16.680
117,MARIA FERNANDA ARIAS OYUELA,79026,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,2336,VENCIDO,MARIA FERNANDA ARIAS OYUELA,MARIA DEL ROCIO FERNANDEZ PORRAS,Devuelto para correcciones,3,2019-11-12 14:21:15.810



RESUMEN POR PRERADICADO:


,preradicado_estado,cantidad
0,NO,119



RESUMEN POR CLASIFICACION DE ASUNTO:


,clasificacion_asunto,cantidad
0,IMPULSO,119



RESUMEN POR ESTADO:


,estado_pelota_caliente,cantidad
0,SIN INICIAR,119


## Revision de datos en general

In [32]:
# =========================================================
# TABLA DINAMICA COMPLETA POR ABOGADO (UN SOLO BLOQUE)
# =========================================================

import pandas as pd

# -----------------------------------------
# 1) BASE DE TRABAJO
# -----------------------------------------
df_td = df_base_final_pelota.copy()

# -----------------------------------------
# 2) RESUMEN GENERAL
# -----------------------------------------
df_resumen_general = (
    df_td.groupby("abogado_normalizado", dropna=False)
    .agg(
        total_filas=("expediente", "count"),
        total_expedientes_unicos=("expediente", "nunique"),
        dias_promedio=("dias_transcurridos", "mean"),
        dias_maximos=("dias_transcurridos", "max")
    )
    .reset_index()
)

# -----------------------------------------
# 3) PIVOT PRERADICADO
# -----------------------------------------
pivot_preradicado = pd.pivot_table(
    df_td,
    index="abogado_normalizado",
    columns="preradicado_estado",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_preradicado.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"preradicado_{c.lower()}"
    for c in pivot_preradicado.columns
]

# -----------------------------------------
# 4) PIVOT CLASIFICACION ASUNTO
# -----------------------------------------
pivot_asunto = pd.pivot_table(
    df_td,
    index="abogado_normalizado",
    columns="clasificacion_asunto",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_asunto.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"asunto_{str(c).lower()}"
    for c in pivot_asunto.columns
]

# -----------------------------------------
# 5) PIVOT ESTADO PELOTA CALIENTE
# -----------------------------------------
pivot_estado = pd.pivot_table(
    df_td,
    index="abogado_normalizado",
    columns="estado_pelota_caliente",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_estado.columns = [
    "abogado_normalizado" if c == "abogado_normalizado"
    else "estado_" + str(c).lower().replace("/", "_").replace(" ", "_")
    for c in pivot_estado.columns
]

# -----------------------------------------
# 6) PIVOT ESTADO DEL DIA
# -----------------------------------------
pivot_dia = pd.pivot_table(
    df_td,
    index="abogado_normalizado",
    columns="estado_del_dia",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_dia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado"
    else "dia_" + str(c).lower().replace(" ", "_")
    for c in pivot_dia.columns
]

# -----------------------------------------
# 7) UNIFICAR TODAS LAS TABLAS
# -----------------------------------------
df_tabla_dinamica_abogado = df_resumen_general.copy()

for tabla in [pivot_preradicado, pivot_asunto, pivot_estado, pivot_dia]:
    df_tabla_dinamica_abogado = df_tabla_dinamica_abogado.merge(
        tabla,
        on="abogado_normalizado",
        how="left"
    )

# -----------------------------------------
# 8) LIMPIEZA FINAL
# -----------------------------------------
columnas_numericas = df_tabla_dinamica_abogado.select_dtypes(include=["number"]).columns
df_tabla_dinamica_abogado[columnas_numericas] = df_tabla_dinamica_abogado[columnas_numericas].fillna(0)

df_tabla_dinamica_abogado["dias_promedio"] = df_tabla_dinamica_abogado["dias_promedio"].round(1)

df_tabla_dinamica_abogado = df_tabla_dinamica_abogado.sort_values(
    by="abogado_normalizado"
).reset_index(drop=True)

# -----------------------------------------
# 9) VISUALIZACION FINAL
# -----------------------------------------
print("TABLA DINAMICA FINAL POR ABOGADO:")
display(df_tabla_dinamica_abogado)

print("\nColumnas generadas:")
print(df_tabla_dinamica_abogado.columns.tolist())

TABLA DINAMICA FINAL POR ABOGADO:


,abogado_normalizado,total_filas,total_expedientes_unicos,dias_promedio,dias_maximos,preradicado_no,asunto_impulso,estado_sin_iniciar,dia_vencido
0,ALVARO ANDRES JIMENEZ VANEGAS,22,22,2622.5,9364,22,22,22,22
1,FRANCY LILIANA DAZA SANCHEZ,37,37,4927.7,17880,37,37,37,37
2,GINA ALEXANDRA ROBERTO TORRES,27,27,2961.2,7226,27,27,27,27
3,ISABEL CRISTINA ANGARITA PERPINAN,14,14,5872.0,11664,14,14,14,14
4,MARIA FERNANDA ARIAS OYUELA,19,19,2863.3,8608,19,19,19,19



Columnas generadas:
['abogado_normalizado', 'total_filas', 'total_expedientes_unicos', 'dias_promedio', 'dias_maximos', 'preradicado_no', 'asunto_impulso', 'estado_sin_iniciar', 'dia_vencido']


# Finalmente agregamos preradicados y tratasmiento de los mismos para ejecutar bases limpias

In [33]:
# =========================================================
# SEGMENTO 9 - PRERADICADOS: FILTRO, LLAVE Y FALTANTES
# =========================================================

import pandas as pd
import numpy as np
import re
import unicodedata

# ---------------------------------------------------------
# 1) FUNCIONES BASE
# ---------------------------------------------------------
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def normalizar_nombre_persona(texto):
    texto = normalizar_texto(texto)
    texto = re.split(r"/|\(", texto)[0].strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def extraer_expediente_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = normalizar_texto(texto)

    patrones = [
        r"NUMERO EXPEDIENTE\s*:\s*(\d+)",
        r"NÚMERO EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s+(\d+)"
    ]

    for patron in patrones:
        m = re.search(patron, texto)
        if m:
            return m.group(1)

    return np.nan

def clasificar_asunto_sidcar(asunto):
    """
    Regla definitiva:
    - si contiene NOTIFIC -> GESTION
    - todo lo demás con expediente -> IMPULSO
    """
    texto = normalizar_texto(asunto)
    if "NOTIFIC" in texto:
        return "GESTION"
    return "IMPULSO"

def clasificar_estado_preradicado(estado):
    texto = normalizar_texto(estado)

    if "DEVUELT" in texto:
        return "DEVUELTO"
    elif "APROB" in texto or "VOBO" in texto or "FIRMA" in texto or "REVISION" in texto:
        return "APROBADO VOBO"
    else:
        return "EN CREACION"

def extraer_regional_desde_oficina(oficina):
    """
    Usa oficina como aproximación de regional.
    """
    if pd.isna(oficina):
        return "SIN REGIONAL"
    texto = normalizar_texto(oficina)
    texto = re.split(r" - |\(|/", texto)[0].strip()
    return texto if texto else "SIN REGIONAL"

def estado_del_dia(dias):
    if pd.isna(dias):
        return "SIN FECHA"
    if dias <= 25:
        return "EN TRAMITE"
    elif 26 <= dias <= 30:
        return "POR VENCER"
    else:
        return "VENCIDO"

# ---------------------------------------------------------
# 2) GRUPO DE ABOGADOS
# ---------------------------------------------------------
NOMBRES_ABOGADOS = [
    "FRANCY LILIANA DAZA SANCHEZ",
    "Maria Fernanda Arias Oyuela",
    "Isabel Cristina Angarita Perpiñán",
    "Gina Alexandra Roberto Torres",
    "Alvaro Andres Jimenez Vanegas",
]

ABOGADOS_OBJETIVO_NORM = [normalizar_nombre_persona(x) for x in NOMBRES_ABOGADOS]

# ---------------------------------------------------------
# 3) MAPEO DE COLUMNAS EN PRERADICADOS
#    A = número
#    C = estado
#    E = fecha
#    G = usuario
#    H = oficina
#    I = asunto
# ---------------------------------------------------------
col_numero_pre  = df_pre.columns[0]
col_estado_pre  = df_pre.columns[2]
col_fecha_pre   = df_pre.columns[4]
col_usuario_pre = df_pre.columns[6]
col_oficina_pre = df_pre.columns[7]
col_asunto_pre  = df_pre.columns[8]

print("Columnas tomadas en preradicados:")
print("Numero   ->", col_numero_pre)
print("Estado   ->", col_estado_pre)
print("Fecha    ->", col_fecha_pre)
print("Usuario  ->", col_usuario_pre)
print("Oficina  ->", col_oficina_pre)
print("Asunto   ->", col_asunto_pre)

# ---------------------------------------------------------
# 4) PREPARAR PRERADICADOS
# ---------------------------------------------------------
df_pre_estado = df_pre.copy()

df_pre_estado["abogado_normalizado"] = df_pre_estado[col_usuario_pre].apply(normalizar_nombre_persona)
df_pre_estado["estado_columna_c"] = df_pre_estado[col_estado_pre].astype(str).fillna("").str.strip()
df_pre_estado["asunto_texto"] = df_pre_estado[col_asunto_pre].astype(str).fillna("").str.strip()
df_pre_estado["fecha_asignacion"] = pd.to_datetime(df_pre_estado[col_fecha_pre], errors="coerce")
df_pre_estado["regional"] = df_pre_estado[col_oficina_pre].apply(extraer_regional_desde_oficina)
df_pre_estado["expediente"] = df_pre_estado["asunto_texto"].apply(extraer_expediente_texto)

# solo el grupo objetivo
df_pre_estado = df_pre_estado[
    df_pre_estado["abogado_normalizado"].isin(ABOGADOS_OBJETIVO_NORM)
].copy()

# solo registros con expediente
df_pre_estado = df_pre_estado[
    df_pre_estado["expediente"].notna()
].copy()

# clasificación asunto
df_pre_estado["clasificacion_asunto"] = df_pre_estado["asunto_texto"].apply(clasificar_asunto_sidcar)

# estado pelota caliente desde columna C
df_pre_estado["estado_pelota_caliente"] = df_pre_estado["estado_columna_c"].apply(clasificar_estado_preradicado)

# preradicado
df_pre_estado["preradicado_estado"] = "SI"

# días
hoy = pd.Timestamp.today().normalize()
df_pre_estado["dias_transcurridos"] = (
    hoy - pd.to_datetime(df_pre_estado["fecha_asignacion"]).dt.normalize()
).dt.days

df_pre_estado["estado_del_dia"] = df_pre_estado["dias_transcurridos"].apply(estado_del_dia)

# llave
df_pre_estado["llave_expediente_asunto"] = (
    df_pre_estado["expediente"].astype(str).str.strip()
    + "||" +
    df_pre_estado["clasificacion_asunto"].astype(str).str.strip()
)

# ---------------------------------------------------------
# 5) PREPARAR LLAVE DE LA TABLA ANTERIOR
# ---------------------------------------------------------
df_tabla_anterior = df_base_final_pelota.copy()

df_tabla_anterior["llave_expediente_asunto"] = (
    df_tabla_anterior["expediente"].astype(str).str.strip()
    + "||" +
    df_tabla_anterior["clasificacion_asunto"].astype(str).str.strip()
)

# ---------------------------------------------------------
# 6) DETECTAR SOLO LOS QUE FALTAN
# ---------------------------------------------------------
llaves_existentes = set(df_tabla_anterior["llave_expediente_asunto"].dropna().unique())

df_pre_faltantes = df_pre_estado[
    ~df_pre_estado["llave_expediente_asunto"].isin(llaves_existentes)
].copy()

print("Total registros válidos en preradicados del grupo:", len(df_pre_estado))
print("Total registros nuevos para agregar:", len(df_pre_faltantes))

print("\nReporte preliminar de faltantes:")
display(
    df_pre_faltantes[
        [
            "abogado_normalizado",
            "expediente",
            "regional",
            "clasificacion_asunto",
            "estado_pelota_caliente",
            "dias_transcurridos",
            "estado_del_dia",
            "estado_columna_c",
            "asunto_texto"
        ]
    ]
)

Columnas tomadas en preradicados:
Numero   -> 3669770
Estado   -> en_borrador
Fecha    -> 20260105_142543290000
Usuario  -> nancy_yaneth_rosas
Oficina  -> dirección_regional_ubaté
Asunto   -> notificación_por_aviso_resolución_dja_no_00961_de_2025_expediente_58013
Total registros válidos en preradicados del grupo: 10
Total registros nuevos para agregar: 10

Reporte preliminar de faltantes:


,abogado_normalizado,expediente,regional,clasificacion_asunto,estado_pelota_caliente,dias_transcurridos,estado_del_dia,estado_columna_c,asunto_texto
3,ISABEL CRISTINA ANGARITA PERPINAN,87770,DIRECCION JURIDICA AMBIENTAL,IMPULSO,APROBADO VOBO,89,VENCIDO,Enviado para Revisión,ELABORACIÓN ITC EXPEDIENTE 87770
169,FRANCY LILIANA DAZA SANCHEZ,2932,DIRECCION JURIDICA AMBIENTAL,IMPULSO,APROBADO VOBO,41,VENCIDO,Enviado para Revisión,Devolución de solicitud de Seguimiento expedie...
1065,MARIA FERNANDA ARIAS OYUELA,34165,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,17,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...
1069,MARIA FERNANDA ARIAS OYUELA,90339,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,17,EN TRAMITE,Enviado para Revisión,Expediente 90339. Citación notificación personal.
1074,MARIA FERNANDA ARIAS OYUELA,98183,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,17,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...
1676,GINA ALEXANDRA ROBERTO TORRES,102448,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,12,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...
1687,GINA ALEXANDRA ROBERTO TORRES,102448,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,12,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...
1693,GINA ALEXANDRA ROBERTO TORRES,102448,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,12,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...
1783,GINA ALEXANDRA ROBERTO TORRES,99977,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,12,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...
1797,GINA ALEXANDRA ROBERTO TORRES,86039,DIRECCION JURIDICA AMBIENTAL,GESTION,APROBADO VOBO,12,EN TRAMITE,Enviado para Revisión,Notificación personal medio electrónico Auto D...


In [34]:
# =========================================================
# SEGMENTO 10 - AGREGAR FALTANTES Y ARMAR BASE FINAL
# =========================================================

# ---------------------------------------------------------
# 1) AJUSTAR FALTANTES AL MISMO FORMATO DE LA BASE FINAL
# ---------------------------------------------------------
df_agregados_preradicado = df_pre_faltantes.copy()

df_agregados_preradicado["responsable_norm"] = df_agregados_preradicado["abogado_normalizado"]
df_agregados_preradicado["remite_norm"] = ""
df_agregados_preradicado["devolucion_sidcar"] = ""
df_agregados_preradicado["estado_sidcar"] = df_agregados_preradicado["estado_columna_c"]

df_agregados_preradicado = df_agregados_preradicado[
    [
        "abogado_normalizado",
        "expediente",
        "regional",
        "preradicado_estado",
        "clasificacion_asunto",
        "estado_pelota_caliente",
        "dias_transcurridos",
        "estado_del_dia",
        "responsable_norm",
        "remite_norm",
        "estado_sidcar",
        "devolucion_sidcar",
        "fecha_asignacion"
    ]
].copy()

# ---------------------------------------------------------
# 2) BASE FINAL ACTUALIZADA
# ---------------------------------------------------------
df_base_final_actualizada = pd.concat(
    [df_base_final_pelota, df_agregados_preradicado],
    ignore_index=True
).copy()

# normalizar y quitar duplicados por llave final
df_base_final_actualizada["llave_expediente_asunto"] = (
    df_base_final_actualizada["expediente"].astype(str).str.strip()
    + "||" +
    df_base_final_actualizada["clasificacion_asunto"].astype(str).str.strip()
)

df_base_final_actualizada = df_base_final_actualizada.drop_duplicates(
    subset=["llave_expediente_asunto"],
    keep="first"
).copy()

df_base_final_actualizada = df_base_final_actualizada.sort_values(
    by=["abogado_normalizado", "expediente", "clasificacion_asunto"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# 3) REPORTE DE QUÉ SE AGREGÓ
# ---------------------------------------------------------
df_reporte_agregados = df_agregados_preradicado.copy()

df_reporte_agregados = df_reporte_agregados.sort_values(
    by=["abogado_normalizado", "expediente", "clasificacion_asunto"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 4) VISUALIZACIÓN
# ---------------------------------------------------------
print("TOTAL REGISTROS AGREGADOS DESDE PRERADICADOS:", len(df_reporte_agregados))
print("\nREPORTE DE AGREGADOS:")
display(df_reporte_agregados)

print("\nBASE FINAL ACTUALIZADA:")
display(df_base_final_actualizada)

print("\nRESUMEN DE AGREGADOS POR ABOGADO:")
display(
    df_reporte_agregados.groupby("abogado_normalizado", dropna=False)
    .size()
    .reset_index(name="cantidad_agregada")
)

print("\nRESUMEN DE AGREGADOS POR ESTADO:")
display(
    df_reporte_agregados.groupby("estado_pelota_caliente", dropna=False)
    .size()
    .reset_index(name="cantidad_agregada")
)

print("\nRESUMEN DE AGREGADOS POR CLASIFICACION:")
display(
    df_reporte_agregados.groupby("clasificacion_asunto", dropna=False)
    .size()
    .reset_index(name="cantidad_agregada")
)

TOTAL REGISTROS AGREGADOS DESDE PRERADICADOS: 10

REPORTE DE AGREGADOS:


,abogado_normalizado,expediente,regional,preradicado_estado,clasificacion_asunto,estado_pelota_caliente,dias_transcurridos,estado_del_dia,responsable_norm,remite_norm,estado_sidcar,devolucion_sidcar,fecha_asignacion
0,FRANCY LILIANA DAZA SANCHEZ,2932,DIRECCION JURIDICA AMBIENTAL,SI,IMPULSO,APROBADO VOBO,41,VENCIDO,FRANCY LILIANA DAZA SANCHEZ,,Enviado para Revisión,,2026-02-23 12:13:08.933
1,GINA ALEXANDRA ROBERTO TORRES,102448,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,12,EN TRAMITE,GINA ALEXANDRA ROBERTO TORRES,,Enviado para Revisión,,2026-03-24 10:00:56.093
2,GINA ALEXANDRA ROBERTO TORRES,102448,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,12,EN TRAMITE,GINA ALEXANDRA ROBERTO TORRES,,Enviado para Revisión,,2026-03-24 10:13:42.320
3,GINA ALEXANDRA ROBERTO TORRES,102448,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,12,EN TRAMITE,GINA ALEXANDRA ROBERTO TORRES,,Enviado para Revisión,,2026-03-24 10:17:50.003
4,GINA ALEXANDRA ROBERTO TORRES,86039,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,12,EN TRAMITE,GINA ALEXANDRA ROBERTO TORRES,,Enviado para Revisión,,2026-03-24 11:38:59.023
5,GINA ALEXANDRA ROBERTO TORRES,99977,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,12,EN TRAMITE,GINA ALEXANDRA ROBERTO TORRES,,Enviado para Revisión,,2026-03-24 11:27:32.493
6,ISABEL CRISTINA ANGARITA PERPINAN,87770,DIRECCION JURIDICA AMBIENTAL,SI,IMPULSO,APROBADO VOBO,89,VENCIDO,ISABEL CRISTINA ANGARITA PERPINAN,,Enviado para Revisión,,2026-01-06 15:27:13.767
7,MARIA FERNANDA ARIAS OYUELA,34165,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 09:27:31.350
8,MARIA FERNANDA ARIAS OYUELA,90339,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 09:49:23.917
9,MARIA FERNANDA ARIAS OYUELA,98183,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 10:29:00.117



BASE FINAL ACTUALIZADA:


,abogado_normalizado,expediente,regional,preradicado_estado,clasificacion_asunto,estado_pelota_caliente,dias_transcurridos,estado_del_dia,responsable_norm,remite_norm,estado_sidcar,devolucion_sidcar,fecha_asignacion,llave_expediente_asunto
0,ALVARO ANDRES JIMENEZ VANEGAS,100651,ALMEIDAS Y MUNICIPIO DE GUATAVITA,NO,IMPULSO,SIN INICIAR,1013,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-06-27 08:11:04.867,100651||IMPULSO
1,ALVARO ANDRES JIMENEZ VANEGAS,101017,SABANA OCCIDENTE,NO,IMPULSO,SIN INICIAR,997,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-07-13 11:16:46.047,101017||IMPULSO
2,ALVARO ANDRES JIMENEZ VANEGAS,101302,ALMEIDAS Y MUNICIPIO DE GUATAVITA,NO,IMPULSO,SIN INICIAR,984,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-07-26 16:24:54.210,101302||IMPULSO
3,ALVARO ANDRES JIMENEZ VANEGAS,102916,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,916,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-10-02 17:12:17.327,102916||IMPULSO
4,ALVARO ANDRES JIMENEZ VANEGAS,102917,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,916,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-10-02 17:12:38.233,102917||IMPULSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,MARIA FERNANDA ARIAS OYUELA,57960,BAJO MAGDALENA,NO,IMPULSO,SIN INICIAR,3484,VENCIDO,MARIA FERNANDA ARIAS OYUELA,WILSON LEONARDO JIMENEZ CASTILLO,nan,nan,2016-09-20 09:44:16.680,57960||IMPULSO
123,MARIA FERNANDA ARIAS OYUELA,79026,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,2336,VENCIDO,MARIA FERNANDA ARIAS OYUELA,MARIA DEL ROCIO FERNANDEZ PORRAS,Devuelto para correcciones,3,2019-11-12 14:21:15.810,79026||IMPULSO
124,MARIA FERNANDA ARIAS OYUELA,90339,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 09:49:23.917,90339||GESTION
125,MARIA FERNANDA ARIAS OYUELA,98183,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 10:29:00.117,98183||GESTION



RESUMEN DE AGREGADOS POR ABOGADO:


,abogado_normalizado,cantidad_agregada
0,FRANCY LILIANA DAZA SANCHEZ,1
1,GINA ALEXANDRA ROBERTO TORRES,5
2,ISABEL CRISTINA ANGARITA PERPINAN,1
3,MARIA FERNANDA ARIAS OYUELA,3



RESUMEN DE AGREGADOS POR ESTADO:


,estado_pelota_caliente,cantidad_agregada
0,APROBADO VOBO,10



RESUMEN DE AGREGADOS POR CLASIFICACION:


,clasificacion_asunto,cantidad_agregada
0,GESTION,8
1,IMPULSO,2


# Analisis general estadistico

In [35]:
# =========================================================
# SEGMENTO 12 - RESUMEN GENERAL POR ABOGADO DE CADA BASE
# =========================================================

import pandas as pd
import numpy as np
import re
import unicodedata

# ---------------------------------------------------------
# 1) FUNCIONES BASE
# ---------------------------------------------------------
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def normalizar_nombre_persona(texto):
    texto = normalizar_texto(texto)
    texto = re.split(r"/|\(", texto)[0].strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def extraer_expediente_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = normalizar_texto(texto)

    patrones = [
        r"NUMERO EXPEDIENTE\s*:\s*(\d+)",
        r"NÚMERO EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s+(\d+)"
    ]

    for patron in patrones:
        m = re.search(patron, texto)
        if m:
            return m.group(1)

    return np.nan

def clasificar_asunto_sidcar(asunto, titulo=""):
    texto = normalizar_texto(f"{asunto} {titulo}")
    if "NOTIFIC" in texto:
        return "GESTION"
    return "IMPULSO"

def extraer_regional_mejor(texto):
    if pd.isna(texto):
        return "SIN REGIONAL"

    texto = normalizar_texto(texto)

    match = re.search(r"DIRECCION\s+REGIONAL\s+([A-Z\s]+)", texto)
    if match:
        regional = match.group(1).strip()
        regional = re.split(r"\(|-|/", regional)[0].strip()
        return regional if regional else "SIN REGIONAL"

    texto_limpio = re.split(r"\(|-|/", texto)[0].strip()

    if len(texto_limpio.split()) >= 3:
        return "SIN REGIONAL"

    return texto_limpio if texto_limpio else "SIN REGIONAL"

# ---------------------------------------------------------
# 2) SI NO EXISTE GRUPO 1, LO RECONSTRUYE
# ---------------------------------------------------------
if "df_grupo_radicados_equipo" not in globals():
    df_rad_equipo = df_rad.copy()

    # expediente
    df_rad_equipo["expediente"] = np.nan
    if "asunto" in df_rad_equipo.columns:
        df_rad_equipo["expediente"] = df_rad_equipo["asunto"].apply(extraer_expediente_texto)

    if "título" in df_rad_equipo.columns:
        mask_rad = df_rad_equipo["expediente"].isna()
        df_rad_equipo.loc[mask_rad, "expediente"] = df_rad_equipo.loc[mask_rad, "título"].apply(extraer_expediente_texto)

    # abogado
    if "elaboró" in df_rad_equipo.columns:
        df_rad_equipo["abogado_normalizado"] = df_rad_equipo["elaboró"].apply(normalizar_nombre_persona)
    else:
        df_rad_equipo["abogado_normalizado"] = "SIN ABOGADO"

    # regional
    if "remitente" in df_rad_equipo.columns:
        df_rad_equipo["regional"] = df_rad_equipo["remitente"].apply(extraer_regional_mejor)
    else:
        df_rad_equipo["regional"] = "SIN REGIONAL"

    # clasificación asunto
    df_rad_equipo["clasificacion_asunto"] = df_rad_equipo.apply(
        lambda row: clasificar_asunto_sidcar(row.get("asunto", ""), row.get("título", "")),
        axis=1
    )

    # fecha
    if "f_radicado" in df_rad_equipo.columns:
        df_rad_equipo["fecha_radicado"] = pd.to_datetime(df_rad_equipo["f_radicado"], errors="coerce")
    else:
        df_rad_equipo["fecha_radicado"] = pd.NaT

    # solo con expediente
    df_rad_equipo = df_rad_equipo[df_rad_equipo["expediente"].notna()].copy()

    # una fila por expediente + clasificación + abogado
    df_rad_equipo["llave_rad"] = (
        df_rad_equipo["expediente"].astype(str).str.strip() + "||" +
        df_rad_equipo["clasificacion_asunto"].astype(str).str.strip() + "||" +
        df_rad_equipo["abogado_normalizado"].astype(str).str.strip()
    )

    df_rad_equipo = df_rad_equipo.drop_duplicates(subset=["llave_rad"], keep="first").copy()

    df_grupo_radicados_equipo = df_rad_equipo[
        [
            "abogado_normalizado",
            "expediente",
            "regional",
            "clasificacion_asunto",
            "fecha_radicado"
        ]
    ].copy()

    df_grupo_radicados_equipo = df_grupo_radicados_equipo.sort_values(
        by=["abogado_normalizado", "expediente", "clasificacion_asunto"]
    ).reset_index(drop=True)

# ---------------------------------------------------------
# 3) SI NO EXISTE GRUPO 2, LO RECONSTRUYE
# ---------------------------------------------------------
if "df_grupo_base_guia" not in globals():
    df_grupo_base_guia = df_base_final_actualizada.copy()
    df_grupo_base_guia = df_grupo_base_guia.sort_values(
        by=["abogado_normalizado", "expediente", "clasificacion_asunto"]
    ).reset_index(drop=True)

# ---------------------------------------------------------
# 4) RESUMEN GRUPO 1 - RADICADOS EQUIPO
# ---------------------------------------------------------
df_resumen_radicados_equipo = (
    df_grupo_radicados_equipo.groupby("abogado_normalizado", dropna=False)
    .agg(
        total_filas=("expediente", "count"),
        total_expedientes_unicos=("expediente", "nunique")
    )
    .reset_index()
)

pivot_asunto_rad = pd.pivot_table(
    df_grupo_radicados_equipo,
    index="abogado_normalizado",
    columns="clasificacion_asunto",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_asunto_rad.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"asunto_{str(c).lower()}"
    for c in pivot_asunto_rad.columns
]

df_resumen_radicados_equipo = df_resumen_radicados_equipo.merge(
    pivot_asunto_rad,
    on="abogado_normalizado",
    how="left"
)

df_resumen_radicados_equipo = df_resumen_radicados_equipo.sort_values(
    by="abogado_normalizado"
).reset_index(drop=True)

# ---------------------------------------------------------
# 5) RESUMEN GRUPO 2 - BASE GUIA AJUSTADA
# ---------------------------------------------------------
df_resumen_base_guia = (
    df_grupo_base_guia.groupby("abogado_normalizado", dropna=False)
    .agg(
        total_filas=("expediente", "count"),
        total_expedientes_unicos=("expediente", "nunique"),
        dias_promedio=("dias_transcurridos", "mean"),
        dias_maximos=("dias_transcurridos", "max")
    )
    .reset_index()
)

pivot_preradicado_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="preradicado_estado",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_preradicado_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"preradicado_{str(c).lower()}"
    for c in pivot_preradicado_guia.columns
]

pivot_asunto_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="clasificacion_asunto",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_asunto_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"asunto_{str(c).lower()}"
    for c in pivot_asunto_guia.columns
]

pivot_estado_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="estado_pelota_caliente",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_estado_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado"
    else "estado_" + str(c).lower().replace("/", "_").replace(" ", "_")
    for c in pivot_estado_guia.columns
]

pivot_dia_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="estado_del_dia",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_dia_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado"
    else "dia_" + str(c).lower().replace(" ", "_")
    for c in pivot_dia_guia.columns
]

for tabla in [pivot_preradicado_guia, pivot_asunto_guia, pivot_estado_guia, pivot_dia_guia]:
    df_resumen_base_guia = df_resumen_base_guia.merge(
        tabla,
        on="abogado_normalizado",
        how="left"
    )

columnas_numericas = df_resumen_base_guia.select_dtypes(include=["number"]).columns
df_resumen_base_guia[columnas_numericas] = df_resumen_base_guia[columnas_numericas].fillna(0)
df_resumen_base_guia["dias_promedio"] = df_resumen_base_guia["dias_promedio"].round(1)

df_resumen_base_guia = df_resumen_base_guia.sort_values(
    by="abogado_normalizado"
).reset_index(drop=True)

# ---------------------------------------------------------
# 6) VISUALIZACION FINAL
# ---------------------------------------------------------
print("RESUMEN GENERAL POR ABOGADO - RADICADOS EQUIPO")
display(df_resumen_radicados_equipo)

print("\nRESUMEN GENERAL POR ABOGADO - BASE GUIA AJUSTADA")
display(df_resumen_base_guia)

RESUMEN GENERAL POR ABOGADO - RADICADOS EQUIPO


,abogado_normalizado,total_filas,total_expedientes_unicos,asunto_gestion,asunto_impulso
0,ALVARO ANDRES JIMENEZ VANEGAS,16,10,6,10
1,FRANCY LILIANA DAZA SANCHEZ,2,2,0,2
2,GINA ALEXANDRA ROBERTO TORRES,24,18,6,18
3,ISABEL CRISTINA ANGARITA PERPINAN,30,30,0,30
4,MARIA FERNANDA ARIAS OYUELA,18,12,6,12



RESUMEN GENERAL POR ABOGADO - BASE GUIA AJUSTADA


,abogado_normalizado,total_filas,total_expedientes_unicos,dias_promedio,dias_maximos,preradicado_no,preradicado_si,asunto_gestion,asunto_impulso,estado_aprobado_vobo,estado_sin_iniciar,dia_en_tramite,dia_vencido
0,ALVARO ANDRES JIMENEZ VANEGAS,22,22,2622.5,9364,22,0,0,22,0,22,0,22
1,FRANCY LILIANA DAZA SANCHEZ,38,38,4799.1,17880,37,1,0,38,1,37,0,38
2,GINA ALEXANDRA ROBERTO TORRES,30,30,2666.3,7226,27,3,3,27,3,27,3,27
3,ISABEL CRISTINA ANGARITA PERPINAN,15,15,5486.5,11664,14,1,0,15,1,14,0,15
4,MARIA FERNANDA ARIAS OYUELA,22,22,2475.2,8608,19,3,3,19,3,19,3,19


In [36]:
# =========================================================
# SEGMENTO 11 - DOS GRUPOS FINALES (CORREGIDO)
# 1) RADICADOS DEL GRUPO DE ABOGADOS
# 2) BASE GUIA AJUSTADA (ETAPAS VS PRERADICADOS)
# =========================================================

import pandas as pd
import numpy as np
import re
import unicodedata

# ---------------------------------------------------------
# 1) ABOGADOS DEL GRUPO
# ---------------------------------------------------------
NOMBRES_ABOGADOS = [
    "FRANCY LILIANA DAZA SANCHEZ",
    "Maria Fernanda Arias Oyuela",
    "Isabel Cristina Angarita Perpiñán",
    "Gina Alexandra Roberto Torres",
    "Alvaro Andres Jimenez Vanegas",
]

# ---------------------------------------------------------
# 2) FUNCIONES BASE
# ---------------------------------------------------------
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def normalizar_nombre_persona(texto):
    texto = normalizar_texto(texto)
    texto = re.split(r"/|\(", texto)[0].strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def extraer_expediente_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = normalizar_texto(texto)

    patrones = [
        r"NUMERO EXPEDIENTE\s*:\s*(\d+)",
        r"NÚMERO EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s*:\s*(\d+)",
        r"EXPEDIENTE\s+(\d+)"
    ]

    for patron in patrones:
        m = re.search(patron, texto)
        if m:
            return m.group(1)

    return np.nan

def clasificar_asunto_sidcar(asunto, titulo=""):
    texto = normalizar_texto(f"{asunto} {titulo}")
    if "NOTIFIC" in texto:
        return "GESTION"
    return "IMPULSO"

def extraer_regional_mejor(texto):
    if pd.isna(texto):
        return "SIN REGIONAL"

    texto = normalizar_texto(texto)

    match = re.search(r"DIRECCION\s+REGIONAL\s+([A-Z\s]+)", texto)
    if match:
        regional = match.group(1).strip()
        regional = re.split(r"\(|-|/", regional)[0].strip()
        return regional if regional else "SIN REGIONAL"

    texto_limpio = re.split(r"\(|-|/", texto)[0].strip()

    # si parece persona, no tomarlo como regional
    if len(texto_limpio.split()) >= 3:
        return "SIN REGIONAL"

    return texto_limpio if texto_limpio else "SIN REGIONAL"

def buscar_columna(df, candidatos):
    cols_norm = {c: normalizar_texto(c) for c in df.columns}
    for candidato in candidatos:
        cand_norm = normalizar_texto(candidato)
        for col_real, col_norm in cols_norm.items():
            if cand_norm == col_norm or cand_norm in col_norm:
                return col_real
    return None

# ---------------------------------------------------------
# 3) NORMALIZAR LISTA DEL GRUPO
# ---------------------------------------------------------
ABOGADOS_GRUPO_NORM = [normalizar_nombre_persona(x) for x in NOMBRES_ABOGADOS]

# ---------------------------------------------------------
# 4) BASE GUIA AJUSTADA
# ---------------------------------------------------------
df_grupo_base_guia = df_base_final_actualizada.copy()

df_grupo_base_guia["expediente"] = df_grupo_base_guia["expediente"].astype(str).str.strip()
df_grupo_base_guia["abogado_normalizado"] = df_grupo_base_guia["abogado_normalizado"].astype(str).str.strip()
df_grupo_base_guia["regional"] = df_grupo_base_guia["regional"].astype(str).str.strip()

df_grupo_base_guia = df_grupo_base_guia.sort_values(
    by=["abogado_normalizado", "expediente", "clasificacion_asunto"]
).reset_index(drop=True)

# catálogo de regional por expediente desde la base guía
df_regional_catalogo = (
    df_grupo_base_guia[["expediente", "regional"]]
    .dropna()
    .copy()
)
df_regional_catalogo["expediente"] = df_regional_catalogo["expediente"].astype(str).str.strip()
df_regional_catalogo["regional"] = df_regional_catalogo["regional"].astype(str).str.strip()
df_regional_catalogo = df_regional_catalogo[
    (df_regional_catalogo["expediente"] != "") &
    (df_regional_catalogo["regional"] != "") &
    (df_regional_catalogo["regional"] != "SIN REGIONAL")
].drop_duplicates(subset=["expediente"], keep="first")

# ---------------------------------------------------------
# 5) GRUPO 1 - RADICADOS DEL GRUPO DE ABOGADOS
# ---------------------------------------------------------
df_rad_equipo = df_rad.copy()

# detectar columnas reales
col_asunto = buscar_columna(df_rad_equipo, ["asunto"])
col_titulo = buscar_columna(df_rad_equipo, ["título", "titulo"])
col_elaboro = buscar_columna(df_rad_equipo, ["elaboró", "elaboro", "usuario", "responsable"])
col_remitente = buscar_columna(df_rad_equipo, ["remitente"])
col_fecha = buscar_columna(df_rad_equipo, ["f_radicado", "fecha radicado", "radicado"])

print("Columnas detectadas en Radicados:")
print("asunto     ->", col_asunto)
print("titulo     ->", col_titulo)
print("elaboro    ->", col_elaboro)
print("remitente  ->", col_remitente)
print("fecha      ->", col_fecha)

# expediente
df_rad_equipo["expediente"] = np.nan
if col_asunto:
    df_rad_equipo["expediente"] = df_rad_equipo[col_asunto].apply(extraer_expediente_texto)

if col_titulo:
    mask_rad = df_rad_equipo["expediente"].isna()
    df_rad_equipo.loc[mask_rad, "expediente"] = df_rad_equipo.loc[mask_rad, col_titulo].apply(extraer_expediente_texto)

# abogado
if col_elaboro:
    df_rad_equipo["abogado_normalizado"] = df_rad_equipo[col_elaboro].apply(normalizar_nombre_persona)
else:
    df_rad_equipo["abogado_normalizado"] = ""

# filtrar solo nuestro grupo
df_rad_equipo = df_rad_equipo[
    df_rad_equipo["abogado_normalizado"].isin(ABOGADOS_GRUPO_NORM)
].copy()

# clasificación asunto
df_rad_equipo["clasificacion_asunto"] = df_rad_equipo.apply(
    lambda row: clasificar_asunto_sidcar(
        row[col_asunto] if col_asunto else "",
        row[col_titulo] if col_titulo else ""
    ),
    axis=1
)

# regional inicial desde remitente
if col_remitente:
    df_rad_equipo["regional"] = df_rad_equipo[col_remitente].apply(extraer_regional_mejor)
else:
    df_rad_equipo["regional"] = "SIN REGIONAL"

# reforzar regional con la base guía por expediente
df_rad_equipo["expediente"] = df_rad_equipo["expediente"].astype(str).str.strip()
df_rad_equipo = df_rad_equipo.merge(
    df_regional_catalogo.rename(columns={"regional": "regional_guia"}),
    on="expediente",
    how="left"
)

df_rad_equipo["regional"] = np.where(
    (df_rad_equipo["regional"].astype(str).str.strip() == "SIN REGIONAL") |
    (df_rad_equipo["regional"].astype(str).str.strip() == ""),
    df_rad_equipo["regional_guia"],
    df_rad_equipo["regional"]
)

df_rad_equipo["regional"] = df_rad_equipo["regional"].fillna("SIN REGIONAL")

# fecha
if col_fecha:
    df_rad_equipo["fecha_radicado"] = pd.to_datetime(df_rad_equipo[col_fecha], errors="coerce")
else:
    df_rad_equipo["fecha_radicado"] = pd.NaT

# solo con expediente
df_rad_equipo = df_rad_equipo[
    df_rad_equipo["expediente"].notna() &
    (df_rad_equipo["expediente"].astype(str).str.strip() != "")
].copy()

# una fila por expediente + clasificación + abogado
df_rad_equipo["llave_rad"] = (
    df_rad_equipo["expediente"].astype(str).str.strip() + "||" +
    df_rad_equipo["clasificacion_asunto"].astype(str).str.strip() + "||" +
    df_rad_equipo["abogado_normalizado"].astype(str).str.strip()
)

df_rad_equipo = df_rad_equipo.drop_duplicates(subset=["llave_rad"], keep="first").copy()

df_grupo_radicados_equipo = df_rad_equipo[
    [
        "abogado_normalizado",
        "expediente",
        "regional",
        "clasificacion_asunto",
        "fecha_radicado"
    ]
].copy()

df_grupo_radicados_equipo = df_grupo_radicados_equipo.sort_values(
    by=["abogado_normalizado", "expediente", "clasificacion_asunto"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 6) VISUALIZACIÓN
# ---------------------------------------------------------
print("Grupo 1 - Radicados del grupo:", df_grupo_radicados_equipo.shape)
print("Grupo 2 - Base guía ajustada:", df_grupo_base_guia.shape)

print("\nVISTA GRUPO 1 - RADICADOS DEL GRUPO")
display(df_grupo_radicados_equipo)

print("\nVISTA GRUPO 2 - BASE GUIA AJUSTADA")
display(df_grupo_base_guia)

Columnas detectadas en Radicados:
asunto     -> asunto
titulo     -> título
elaboro    -> elaboró
remitente  -> remitente
fecha      -> f_radicado
Grupo 1 - Radicados del grupo: (90, 5)
Grupo 2 - Base guía ajustada: (127, 14)

VISTA GRUPO 1 - RADICADOS DEL GRUPO


,abogado_normalizado,expediente,regional,clasificacion_asunto,fecha_radicado
0,ALVARO ANDRES JIMENEZ VANEGAS,102134,TEQUENDAMA,GESTION,2026-03-13 11:20:27.977
1,ALVARO ANDRES JIMENEZ VANEGAS,102134,TEQUENDAMA,IMPULSO,2026-03-05 16:20:01.730
2,ALVARO ANDRES JIMENEZ VANEGAS,33512,SABANA OCCIDENTE,IMPULSO,2026-03-27 09:56:02.127
3,ALVARO ANDRES JIMENEZ VANEGAS,57966,TEQUENDAMA,IMPULSO,2026-03-10 13:20:18.303
4,ALVARO ANDRES JIMENEZ VANEGAS,58028,TEQUENDAMA,IMPULSO,2026-03-10 12:30:18.723
...,...,...,...,...,...
85,MARIA FERNANDA ARIAS OYUELA,98183,TEQUENDAMA,IMPULSO,2026-03-10 17:57:40.087
86,MARIA FERNANDA ARIAS OYUELA,98496,ALMEIDAS Y GUATAVITA,GESTION,2026-03-13 09:58:51.790
87,MARIA FERNANDA ARIAS OYUELA,98496,ALMEIDAS Y GUATAVITA,IMPULSO,2026-03-10 13:52:13.243
88,MARIA FERNANDA ARIAS OYUELA,99975,TEQUENDAMA,IMPULSO,2026-03-25 22:15:39.803



VISTA GRUPO 2 - BASE GUIA AJUSTADA


,abogado_normalizado,expediente,regional,preradicado_estado,clasificacion_asunto,estado_pelota_caliente,dias_transcurridos,estado_del_dia,responsable_norm,remite_norm,estado_sidcar,devolucion_sidcar,fecha_asignacion,llave_expediente_asunto
0,ALVARO ANDRES JIMENEZ VANEGAS,100651,ALMEIDAS Y MUNICIPIO DE GUATAVITA,NO,IMPULSO,SIN INICIAR,1013,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-06-27 08:11:04.867,100651||IMPULSO
1,ALVARO ANDRES JIMENEZ VANEGAS,101017,SABANA OCCIDENTE,NO,IMPULSO,SIN INICIAR,997,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-07-13 11:16:46.047,101017||IMPULSO
2,ALVARO ANDRES JIMENEZ VANEGAS,101302,ALMEIDAS Y MUNICIPIO DE GUATAVITA,NO,IMPULSO,SIN INICIAR,984,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-07-26 16:24:54.210,101302||IMPULSO
3,ALVARO ANDRES JIMENEZ VANEGAS,102916,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,916,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-10-02 17:12:17.327,102916||IMPULSO
4,ALVARO ANDRES JIMENEZ VANEGAS,102917,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,916,VENCIDO,ANGI TATIANA CUERVO DAZA,ALVARO ANDRES JIMENEZ VANEGAS,nan,nan,2023-10-02 17:12:38.233,102917||IMPULSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,MARIA FERNANDA ARIAS OYUELA,57960,BAJO MAGDALENA,NO,IMPULSO,SIN INICIAR,3484,VENCIDO,MARIA FERNANDA ARIAS OYUELA,WILSON LEONARDO JIMENEZ CASTILLO,nan,nan,2016-09-20 09:44:16.680,57960||IMPULSO
123,MARIA FERNANDA ARIAS OYUELA,79026,TEQUENDAMA,NO,IMPULSO,SIN INICIAR,2336,VENCIDO,MARIA FERNANDA ARIAS OYUELA,MARIA DEL ROCIO FERNANDEZ PORRAS,Devuelto para correcciones,3,2019-11-12 14:21:15.810,79026||IMPULSO
124,MARIA FERNANDA ARIAS OYUELA,90339,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 09:49:23.917,90339||GESTION
125,MARIA FERNANDA ARIAS OYUELA,98183,DIRECCION JURIDICA AMBIENTAL,SI,GESTION,APROBADO VOBO,17,EN TRAMITE,MARIA FERNANDA ARIAS OYUELA,,Enviado para Revisión,,2026-03-19 10:29:00.117,98183||GESTION


In [37]:
# =========================================================
# SEGMENTO 12 - RESUMEN GENERAL POR ABOGADO DE CADA BASE
# =========================================================

# ---------------------------------------------------------
# 1) RESUMEN GRUPO 1 - RADICADOS EQUIPO
# ---------------------------------------------------------
df_resumen_radicados_equipo = (
    df_grupo_radicados_equipo.groupby("abogado_normalizado", dropna=False)
    .agg(
        total_filas=("expediente", "count"),
        total_expedientes_unicos=("expediente", "nunique")
    )
    .reset_index()
)

pivot_asunto_rad = pd.pivot_table(
    df_grupo_radicados_equipo,
    index="abogado_normalizado",
    columns="clasificacion_asunto",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_asunto_rad.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"asunto_{str(c).lower()}"
    for c in pivot_asunto_rad.columns
]

df_resumen_radicados_equipo = df_resumen_radicados_equipo.merge(
    pivot_asunto_rad,
    on="abogado_normalizado",
    how="left"
)

df_resumen_radicados_equipo = df_resumen_radicados_equipo.sort_values(
    by="abogado_normalizado"
).reset_index(drop=True)

# ---------------------------------------------------------
# 2) RESUMEN GRUPO 2 - BASE GUIA AJUSTADA
# ---------------------------------------------------------
df_resumen_base_guia = (
    df_grupo_base_guia.groupby("abogado_normalizado", dropna=False)
    .agg(
        total_filas=("expediente", "count"),
        total_expedientes_unicos=("expediente", "nunique"),
        dias_promedio=("dias_transcurridos", "mean"),
        dias_maximos=("dias_transcurridos", "max")
    )
    .reset_index()
)

pivot_preradicado_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="preradicado_estado",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_preradicado_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"preradicado_{str(c).lower()}"
    for c in pivot_preradicado_guia.columns
]

pivot_asunto_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="clasificacion_asunto",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_asunto_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado" else f"asunto_{str(c).lower()}"
    for c in pivot_asunto_guia.columns
]

pivot_estado_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="estado_pelota_caliente",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_estado_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado"
    else "estado_" + str(c).lower().replace("/", "_").replace(" ", "_")
    for c in pivot_estado_guia.columns
]

pivot_dia_guia = pd.pivot_table(
    df_grupo_base_guia,
    index="abogado_normalizado",
    columns="estado_del_dia",
    values="expediente",
    aggfunc="count",
    fill_value=0
).reset_index()

pivot_dia_guia.columns = [
    "abogado_normalizado" if c == "abogado_normalizado"
    else "dia_" + str(c).lower().replace(" ", "_")
    for c in pivot_dia_guia.columns
]

for tabla in [pivot_preradicado_guia, pivot_asunto_guia, pivot_estado_guia, pivot_dia_guia]:
    df_resumen_base_guia = df_resumen_base_guia.merge(
        tabla,
        on="abogado_normalizado",
        how="left"
    )

columnas_numericas = df_resumen_base_guia.select_dtypes(include=["number"]).columns
df_resumen_base_guia[columnas_numericas] = df_resumen_base_guia[columnas_numericas].fillna(0)
df_resumen_base_guia["dias_promedio"] = df_resumen_base_guia["dias_promedio"].round(1)

df_resumen_base_guia = df_resumen_base_guia.sort_values(
    by="abogado_normalizado"
).reset_index(drop=True)

# ---------------------------------------------------------
# 3) VISUALIZACION FINAL
# ---------------------------------------------------------
print("RESUMEN GENERAL POR ABOGADO - RADICADOS EQUIPO")
display(df_resumen_radicados_equipo)

print("\nRESUMEN GENERAL POR ABOGADO - BASE GUIA AJUSTADA")
display(df_resumen_base_guia)

RESUMEN GENERAL POR ABOGADO - RADICADOS EQUIPO


,abogado_normalizado,total_filas,total_expedientes_unicos,asunto_gestion,asunto_impulso
0,ALVARO ANDRES JIMENEZ VANEGAS,16,10,6,10
1,FRANCY LILIANA DAZA SANCHEZ,2,2,0,2
2,GINA ALEXANDRA ROBERTO TORRES,24,18,6,18
3,ISABEL CRISTINA ANGARITA PERPINAN,30,30,0,30
4,MARIA FERNANDA ARIAS OYUELA,18,12,6,12



RESUMEN GENERAL POR ABOGADO - BASE GUIA AJUSTADA


,abogado_normalizado,total_filas,total_expedientes_unicos,dias_promedio,dias_maximos,preradicado_no,preradicado_si,asunto_gestion,asunto_impulso,estado_aprobado_vobo,estado_sin_iniciar,dia_en_tramite,dia_vencido
0,ALVARO ANDRES JIMENEZ VANEGAS,22,22,2622.5,9364,22,0,0,22,0,22,0,22
1,FRANCY LILIANA DAZA SANCHEZ,38,38,4799.1,17880,37,1,0,38,1,37,0,38
2,GINA ALEXANDRA ROBERTO TORRES,30,30,2666.3,7226,27,3,3,27,3,27,3,27
3,ISABEL CRISTINA ANGARITA PERPINAN,15,15,5486.5,11664,14,1,0,15,1,14,0,15
4,MARIA FERNANDA ARIAS OYUELA,22,22,2475.2,8608,19,3,3,19,3,19,3,19


# Segmento final - exportar a excel

In [38]:
# =========================================================
# EXPORTAR EXCEL FINAL A output_SIDCAR
# =========================================================

from pathlib import Path

# ---------------------------------------------------------
# 1) CREAR CARPETA SI NO EXISTE
# ---------------------------------------------------------
ruta_salida = Path("./output_SIDCAR")
ruta_salida.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) NOMBRE DEL ARCHIVO
# ---------------------------------------------------------
ruta_archivo = ruta_salida / "seguimiento_sae_sidcar_grupo_A.xlsx"

# ---------------------------------------------------------
# 3) EXPORTAR A EXCEL CON DOS HOJAS
# ---------------------------------------------------------
with pd.ExcelWriter(ruta_archivo, engine="openpyxl") as writer:
    
    # Hoja 1: Radicados equipo
    df_grupo_radicados_equipo.to_excel(
        writer,
        sheet_name="Radicados_Equipo",
        index=False
    )
    
    # Hoja 2: Base guía ajustada
    df_grupo_base_guia.to_excel(
        writer,
        sheet_name="Base_Guia",
        index=False
    )

# ---------------------------------------------------------
# 4) CONFIRMACIÓN
# ---------------------------------------------------------
print("Archivo generado correctamente en:")
print(ruta_archivo.resolve())

Archivo generado correctamente en:
C:\Users\JULIANCHO ROBLES\Desktop\PROYECTOS GITHUB\oficios_notificaciones\output_SIDCAR\seguimiento_sae_sidcar_grupo_A.xlsx
